# 06 — Embedding Generation

Generate normalized float32 embeddings from complete chunk text, including the Section prefix. The version 2 run manifest records ordered IDs and text digests, metadata/vector file hashes, dimension, normalization and model revision. This exploratory output is not yet a production stage checkpoint; notebook 07 validates and rebuilds unsigned stages.


In [ ]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "intelligent_scada").is_dir():
            return candidate
    raise RuntimeError("Could not locate the Intelligent SCADA project root.")

project_root = find_project_root(Path.cwd().resolve())
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from intelligent_scada.config import Settings

settings = Settings.from_env(project_root=project_root, env_override=True).for_default_document()
settings.ensure_output_dirs()


In [ ]:
from intelligent_scada.artifacts import read_json
from intelligent_scada.embeddings.embedding_service import EmbeddingService

chunks = read_json(settings.text_chunks_path)
service = EmbeddingService(
    settings.embedding_model_name,
    expected_dimension=settings.embedding_dimension,
    batch_size=settings.embedding_batch_size,
    hf_token=settings.hf_token,
    revision=settings.embedding_model_revision,
)
embeddings = service.embed_chunks(
    chunks,
    settings.embeddings_npy_path,
    settings.embeddings_meta_path,
    settings.embeddings_manifest_path,
)

print(f"Shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
print(f"Output: {settings.embeddings_npy_path}")
